In [1]:
import faiss

In [2]:
import torch

In [3]:
# pip install pymupdf

In [4]:
# pip install superkmeans


In [31]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [5]:
print("Torch version:", torch.__version__)
print("Built with CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.11.0+cu128
Built with CUDA: 12.8
CUDA available: True
GPU count: 1
GPU: NVIDIA GeForce GTX 1650


In [6]:
from sentence_transformers import SentenceTransformer

D:\Anaconda3\envs\financial_analysis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# model = SentenceTransformer(
#     "sentence-transformers/all-MiniLM-L6-v2"
# )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1378.90it/s]


In [5]:
# model = SentenceTransformer(
#     "sentence-transformers/all-MiniLM-L6-v2",
#     device="cuda"
# )

# print(model.device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 248.64it/s]


cuda:0


In [6]:
# %%time
# texts = ["hello how", "are you"]
# embeddings = model.encode(
#     texts,
#     batch_size=32,
#     show_progress_bar=True,
#     convert_to_numpy=True
# )

Batches: 100%|██████████| 1/1 [00:08<00:00,  8.00s/it]

CPU times: total: 438 ms
Wall time: 9.01 s


In [7]:
# embeddings.shape

(2, 384)

In [8]:
# from langchain_community.document_loaders import PyPDFLoader

C:\Users\ekawa\AppData\Local\Temp\ipykernel_12984\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
pdf_files = [r"D:\RAG\Daniel Vaughan - Data Science_ The Hard Parts_ Techniques for Excelling at Data Science-O'Reilly Media (2023).pdf"]

In [38]:
# from pypdf import PdfReader

# docs = []

# for pdf in pdf_files:
#     reader = PdfReader(pdf)
#     for page in reader.pages:
#         text = page.extract_text()
#         if text:  # Ensures empty pages are skipped
#             docs.append(text)

In [39]:
# from langchain_community.document_loaders import PyPDFLoader

# docs = []

# for pdf in pdf_files:
#     loader = PyPDFLoader(pdf)
#     docs.extend(loader.load())

# Reading document by each page

In [8]:
import fitz
from langchain_core.documents import Document
from pathlib import Path

docs = []

for pdf_path in Path(r"D:\RAG").glob("*.pdf"):
    pdf = fitz.open(pdf_path)

    for page_num, page in enumerate(pdf):
        docs.append(
            Document(
                page_content=page.get_text(),
                metadata={
                    "source": str(pdf_path),
                    "page": page_num + 1,
                },
            )
        )

In [9]:
docs[0]

Document(metadata={'source': "D:\\RAG\\Daniel Vaughan - Data Science_ The Hard Parts_ Techniques for Excelling at Data Science-O'Reilly Media (2023).pdf", 'page': 1}, page_content='Daniel Vaughan\nData Science: \n The Hard Parts\nTechniques for Excelling at Data Science\n')

In [10]:
len(docs)

257

# Chunking 

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

In [16]:
len(chunks)

1222

In [12]:
chunks[0]

Document(metadata={'source': "D:\\RAG\\Daniel Vaughan - Data Science_ The Hard Parts_ Techniques for Excelling at Data Science-O'Reilly Media (2023).pdf", 'page': 1}, page_content='Daniel Vaughan\nData Science: \n The Hard Parts\nTechniques for Excelling at Data Science')

# Creating embeddings

In [13]:
%%time
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

texts = [c.page_content for c in chunks]

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches: 100%|██████████| 39/39 [00:08<00:00,  4.52it/s]

CPU times: total: 4.48 s
Wall time: 19.2 s


In [28]:
embeddings[0]

array([-8.75615031e-02, -5.96766733e-03,  1.81709956e-02,  2.98746750e-02,
       -1.72686763e-02, -8.13431144e-02, -5.85161373e-02,  2.26520031e-04,
       -1.11557126e-01,  3.24641727e-02, -4.86090519e-02, -2.61192899e-02,
        5.59816957e-02, -3.59959565e-02, -4.86473776e-02,  4.93356884e-02,
       -5.82497381e-02,  2.59290216e-03, -4.71235216e-02, -4.74932343e-02,
       -6.72077388e-02,  1.31617384e-02, -3.15683223e-02, -2.50537898e-02,
        9.33796242e-02,  8.90773628e-03,  6.92704767e-02,  1.91081036e-02,
        5.91343679e-02, -2.86800563e-02, -4.07486707e-02,  3.40158567e-02,
       -4.36567748e-03,  6.59036487e-02, -3.99589427e-02, -3.32262949e-04,
        5.10920733e-02,  4.37978804e-02,  3.47395092e-02,  4.48297486e-02,
        1.22288624e-02, -4.62021939e-02,  1.71295516e-02, -7.73172872e-03,
       -7.96461385e-03, -4.03011851e-02, -1.66503023e-02, -6.89346939e-02,
        1.33233834e-02,  1.66164134e-02, -1.33411095e-01,  1.39883077e-02,
       -1.53298024e-02,  

# Saving embeddings in FAISS vector store

In [14]:
# import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [17]:
metadata = []

for chunk in chunks:
    metadata.append({
        "text": chunk.page_content,
        "source": chunk.metadata["source"],
        "page": chunk.metadata["page"]
    })

In [18]:
import pickle

with open("metadata.pkl","wb") as f:
    pickle.dump(metadata,f)

In [19]:
faiss.write_index(index, "docs.index")

# Retrieval Block

In [22]:
query = "What are some desirable metric properties?"

query_vector = model.encode(
    [query],
    convert_to_numpy=True
)
# query_vector

In [25]:
D, I = index.search(query_vector, k=5)
results = [metadata[i] for i in I[0]]
results

[{'text': 'Desirable properties that good metrics should have.\nA good metric must be measurable, actionable, relevant, and timely.\nDecomposing metrics into submetrics allows you to improve on these properties.\nFunnel-type decompositions are easy to use, and once you get used to them,\nyou’ll start to see funnels everywhere.\nA simple trick of multiplying and dividing by one metric can take you very far.\nBut the choice of that metric is far from obvious, and you need good knowledge\nof the business to find it.',
  'source': "D:\\RAG\\Daniel Vaughan - Data Science_ The Hard Parts_ Techniques for Excelling at Data Science-O'Reilly Media (2023).pdf",
  'page': 35},
 {'text': 'Key Takeaways                                                                                                                  9\nFurther Reading                                                                                                             10\n2. Metrics Design. . . . . . . . . . . . . . . . . . . . 

In [29]:
query = "What are some good to have metric properties?"

query_vector = model.encode(
    [query],
    convert_to_numpy=True
)
# query_vector
D, I = index.search(query_vector, k=5)
results = [metadata[i] for i in I[0]]
results

[{'text': 'Desirable properties that good metrics should have.\nA good metric must be measurable, actionable, relevant, and timely.\nDecomposing metrics into submetrics allows you to improve on these properties.\nFunnel-type decompositions are easy to use, and once you get used to them,\nyou’ll start to see funnels everywhere.\nA simple trick of multiplying and dividing by one metric can take you very far.\nBut the choice of that metric is far from obvious, and you need good knowledge\nof the business to find it.',
  'source': "D:\\RAG\\Daniel Vaughan - Data Science_ The Hard Parts_ Techniques for Excelling at Data Science-O'Reilly Media (2023).pdf",
  'page': 35},
 {'text': 'Metrics design is critical if your aim is to find levers that can drive actions. I have\nreverse engineered the problem to arrive at some desirable properties for metrics\ndesign.\n16 \n| \nChapter 2: Metrics Design',
  'source': "D:\\RAG\\Daniel Vaughan - Data Science_ The Hard Parts_ Techniques for Excelling at 

In [30]:
context = "\n\n".join(
    chunk["text"]
    for chunk in results
)
context

'Desirable properties that good metrics should have.\nA good metric must be measurable, actionable, relevant, and timely.\nDecomposing metrics into submetrics allows you to improve on these properties.\nFunnel-type decompositions are easy to use, and once you get used to them,\nyou’ll start to see funnels everywhere.\nA simple trick of multiplying and dividing by one metric can take you very far.\nBut the choice of that metric is far from obvious, and you need good knowledge\nof the business to find it.\n\nMetrics design is critical if your aim is to find levers that can drive actions. I have\nreverse engineered the problem to arrive at some desirable properties for metrics\ndesign.\n16 \n| \nChapter 2: Metrics Design\n\nKey Takeaways                                                                                                                  9\nFurther Reading                                                                                                             10\n2. Metrics 

In [34]:
context = ""

for r in results:
    context += f"""
Source: {r['source']}
Page: {r['page']}

{r['text']}

----------------
"""
context

"\nSource: D:\\RAG\\Daniel Vaughan - Data Science_ The Hard Parts_ Techniques for Excelling at Data Science-O'Reilly Media (2023).pdf\nPage: 35\n\nDesirable properties that good metrics should have.\nA good metric must be measurable, actionable, relevant, and timely.\nDecomposing metrics into submetrics allows you to improve on these properties.\nFunnel-type decompositions are easy to use, and once you get used to them,\nyou’ll start to see funnels everywhere.\nA simple trick of multiplying and dividing by one metric can take you very far.\nBut the choice of that metric is far from obvious, and you need good knowledge\nof the business to find it.\n\n----------------\n\nSource: D:\\RAG\\Daniel Vaughan - Data Science_ The Hard Parts_ Techniques for Excelling at Data Science-O'Reilly Media (2023).pdf\nPage: 34\n\nMetrics design is critical if your aim is to find levers that can drive actions. I have\nreverse engineered the problem to arrive at some desirable properties for metrics\ndesign

# Generation

In [32]:
from ollama import chat

In [33]:
prompt = f"""
You are a helpful assistant answering questions from documents.

Rules:
- Answer only from the provided context.
- If the answer is not present, say "I could not find that information."
- Quote important facts when possible.
- Mention the source document if available.

Context:
{context}

Question:
{query}

Answer:
"""

In [35]:
response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response["message"]["content"])

According to the provided context:

Desirable Properties That Good Metrics Should Have are:

1. Measurable: The metric should be able to be measured or quantified.
2. Actionable: The metric should provide a clear direction for action.

Additionally, the text mentions two other desirable properties:

3. Relevant: The metric should be informative for the problem at hand. (Source: "Relevance is the property of having the right metric for the right problem.")
4. Timely: Good metrics drive actions when you need them to.
